# 1. Setup Groq & Gemini APIs
Instead of OpenAI, we are using two alternative platforms that offer free tiers:

**Groq Cloud:** Used for fast text-based extraction from Markdown. Create a free account and get an API key at [console.groq.com.](https://console.groq.com/home)

**Google Gemini:** Used for multimodal (Vision) extraction from page images. Get a free API key at https://aistudio.google.com/api-keys

Both keys will be entered securely using `getpass` to keep them hidden. This setup allows for high-quality document analysis without any API costs.

# 2. Import Required Libraries

Import **Docling** helpers to handle the conversion of PDF documents into structured **Markdown text** and **high-quality page images**. Instead of OpenAI, we import the **Groq** library for lightning-fast text extraction and the **Google Generative AI** library to leverage **Gemini's vision capabilities** for image-based analysis. We also use `getpass` to securely handle **API keys** without exposing them in the code.

In [1]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
import groq
import google.generativeai as genai
import getpass

c:\Users\ADmiN\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ADmiN\AppData\Local\Temp\ipykernel_1176\3115271598.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# Define PDF conversion options
pipeline_options = PdfPipelineOptions()
pipeline_options.images_scale = 2
pipeline_options.generate_page_images = True

# Initialize the converter with format options
doc_converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

# 3. Load PDF Document

Specify the path to the PDF invoice to extract. Use the configured `DocumentConverter` to convert the file into a structured `document` containing both text and page images.
The converted document is used for subsequent text and image-based extraction steps.

In [3]:
# Specify the path to your PDF file
pdf_path = "./docs/pdf_invoice.pdf"
result = doc_converter.convert(pdf_path)

[INFO] 2026-03-24 10:17:51,191 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-24 10:17:51,242 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\ADmiN\AppData\Local\Programs\Python\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-24 10:17:51,247 [RapidOCR] main.py:53: Using C:\Users\ADmiN\AppData\Local\Programs\Python\Python312\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-24 10:17:51,659 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-24 10:17:51,671 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\ADmiN\AppData\Local\Programs\Python\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-24 10:17:51,671 [RapidOCR] main.py:53: Using C:\Users\ADmiN\AppData\Local\Programs\Python\Python312\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-24 10:17:51,870 [RapidOCR] base.py:

In [4]:
# 1. Get the image safely from the result object
try:
    # Get the first page from the result list
    first_page = result.pages[0] 
    image_page = first_page.image # Docling image is already PIL compatible
    print("✅ Page image loaded successfully!")
except NameError:
    print("❌ Error: Make sure you ran the cell where 'result' is defined!")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Page image loaded successfully!


# 4. Setup Extraction Helpers
Extract the document text from the converted `document` and export it to **Markdown** format for efficient LLM consumption. Since we are using a hybrid approach, we will create two specific helper functions: one for **Groq** to handle fast text-based extraction, and another for **Gemini** to handle visual extraction from page images.

In [5]:
# Extract text from the document
doc = result.document
text = doc.export_to_markdown()

In [11]:
# Setup Groq (Text)
groq_key = getpass.getpass("Enter Groq API Key: ")
groq_client = groq.Groq(api_key=groq_key)

# Setup Gemini (Vision) - THE GUARANTEED WAY
gemini_key = getpass.getpass("Enter Gemini API Key: ")
genai.configure(api_key=gemini_key)

# كنستعملو هاد السمية الكاملة باش نتفاداو 404 نهائياً
gemini_model = genai.GenerativeModel('models/gemini-1.5-flash') 

print("✅ Gemini Model is set up with the full path!")

✅ Gemini Model is set up with the full path!


In [18]:
# Fonction l-Groq (Text)
def ask_groq_text(user_content):
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant", 
        messages=[{"role": "user", "content": user_content}]
    )
    return response.choices[0].message.content

def ask_gemini_vision(image_pil, prompt):
    """
    Finds an available vision model and generates content based on an image and a prompt.
    """
    # 1. List all models that support content generation
    available_models = [
        m.name for m in genai.list_models() 
        if 'generateContent' in m.supported_generation_methods
    ]
    # 2. Select the best available 'flash' model for speed, fallback to the first available
    chosen_model_name = next(
        (m for m in available_models if 'flash' in m), 
        available_models[0]
    )
    print(f"🔄 Using model: {chosen_model_name}")
    # 3. Initialize the model and generate the response
    model = genai.GenerativeModel(model_name=chosen_model_name)
    response = model.generate_content([prompt, image_pil])
    
    return response.text

print("✅ ask_gemini_vision function is ready!")

✅ ask_gemini_vision function is ready!


# 5. Extract Attributes Using Document Text (Markdown)
Use the exported **Markdown text** as input to the LLM and ask focused questions to extract invoice attributes such as `Invoice Number` and `Balance Due`. Instead of GPT, we use the **Llama-3-8b** model on **Groq** for near-instant text analysis and extraction.

In [16]:
# Use Groq to find the invoice number within the Markdown text
question_1 = f"What is the invoice number in the following text? {text}"
answer_1 = ask_groq_text(question_1)
print(f"Groq Answer (Invoice Number): {answer_1}")

Groq Answer (Invoice Number): The invoice number is # 16384.


In [9]:
# Use Groq to find the total balance due
question_2 = f"What is the total balance due in the following text? {text}"
answer_2 = ask_groq_text(question_2)
print(f"Groq Answer (Balance Due): {answer_2}")

Groq Answer (Balance Due): The total balance due is $6,208.84.


# 6. Extract Attributes Using Page Images
Instead of manual OCR or complex encoding, we pass the **page images** generated by Docling directly to **Gemini 1.5 Flash**. This multimodal approach allows the model to "see" the document layout, which is essential for extracting visually positioned information like the `Invoice Date` or `Shipping Address`. By using the direct **PIL image** format, we simplify the pipeline and improve extraction accuracy.

In [19]:
# --- 1. Get Image ---
first_page = result.pages[0]
image_page = first_page.image

# --- 2. Extract Date ---
question_date = "Extract the invoice date from the image"
answer_date = ask_gemini_vision(image_page, question_date)
print(f"Gemini Answer (Date): {answer_date}")

# --- 3. Extract Address ---
question_address = "What is the shipping address on the invoice?"
answer_address = ask_gemini_vision(image_page, question_address)
print(f"Gemini Answer (Address): {answer_address}")

🔄 Using model: models/gemini-2.5-flash
Gemini Answer (Date): Dec 08 2012
🔄 Using model: models/gemini-2.5-flash
Gemini Answer (Address): The shipping address on the invoice is:
Nottingham, England, United Kingdom


# 7. Key Considerations

When building AI-based extraction systems, the choice of document representation is critical. For this project, we moved beyond a single-provider model to a **Hybrid Architecture**:

* **Markdown for Text**: By using **Groq (Llama-3)**, we achieve near-instant extraction for structured data like `Invoice Numbers` at zero cost.
* **Direct Vision for Layout**: By passing **PIL images** directly to **Gemini 1.5 Flash**, we avoid complex `base64` encoding and allow the model to understand visual positioning for fields like `Shipping Address`.
* **Optimization**: The best decision for your engine depends on the balance between **Speed** (Groq), **Visual Intelligence** (Gemini), and **Cost** (using free tiers).